# 7장. RAG 시스템 평가 — 04. 커스텀 평가자 (Custom Evaluators)

## 학습 목표

- 커스텀 평가자 함수의 시그니처와 반환 형식 이해
- **규칙 기반**, **LLM 기반**, **임베딩 기반** 세 가지 커스텀 평가자 직접 구현
- 여러 평가자를 조합해 다차원 평가 수행

## 사전 지식

- 03번 노트북에서 LangSmith 기본 평가 경험
- 커스텀 평가자 함수 시그니처 이해

## 커스텀 평가자 기본 구조

모든 커스텀 평가자는 다음 형식을 따라야 해요:

```python
from langsmith.schemas import Run, Example

def my_evaluator(run: Run, example: Example) -> dict:
    # run.outputs  : 시스템이 생성한 출력 (평가 대상)
    # example.inputs  : 데이터셋의 입력
    # example.outputs : 데이터셋의 참조 답변 (선택적)
    
    score = ...  # 0~1 사이 값 계산
    return {"key": "metric_name", "score": score}
```

- `key`: LangSmith 대시보드에 표시될 지표 이름
- `score`: 0~1 사이 숫자 (또는 불리언 0/1)

---

## 환경 설정

In [ ]:
# 필요한 패키지 설치
# !pip install -qU langsmith langchain langchain-openai

In [ ]:
from dotenv import load_dotenv
import os

# 환경변수 로드
load_dotenv()

# LangSmith 설정 확인
print(f"LANGCHAIN_API_KEY: {'✅ 설정됨' if os.getenv('LANGCHAIN_API_KEY') else '❌ 미설정'}")
print(f"LANGCHAIN_TRACING_V2: {os.getenv('LANGCHAIN_TRACING_V2', 'false')}")
print(f"LANGCHAIN_PROJECT: {os.getenv('LANGCHAIN_PROJECT', 'default')}")


---

## 평가할 RAG 시스템 구축

In [ ]:
# ---------------------------------------------------
# RAG 시스템 구축 (벡터 DB + 체인)
# ---------------------------------------------------
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1단계: 문서 로드 및 분할
loader = PyMuPDFLoader("data/attention_paper.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_docs = text_splitter.split_documents(docs)

# 2단계: 벡터 DB 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever()

# 3단계: 프롬프트 정의
prompt = PromptTemplate.from_template(
    """Answer the question based on the context provided. Be concise and specific.

Context: {context}

Question: {question}

Answer:"""
)

# 4단계: LLM 및 체인 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG 시스템 구축 완료")

### 평가용 함수 정의

LangSmith `evaluate()` 함수는 `inputs -> outputs` 형식을 요구해요. RAG 체인의 답변뿐 아니라 **검색된 컨텍스트**도 함께 반환해서 평가자가 활용할 수 있도록 합니다.

In [ ]:
# ---------------------------------------------------
# 평가용 함수 정의 — 질문, 답변, 컨텍스트 모두 반환
# ---------------------------------------------------
# ============================================================
# TODO: 질문, 답변, 컨텍스트를 모두 반환하는 평가용 함수를 완성하세요
# 힌트: retriever.invoke(question)으로 docs를 검색하고, rag_chain.invoke(question)으로 답변 생성
# 예상 결과: {"question": ..., "answer": ..., "context": ...} 형식
# ============================================================



---

## 1. 규칙 기반 커스텀 평가자

**LLM 호출 없이** 간단한 규칙으로 답변을 평가해요. 빠르고 비용이 들지 않아서 개발 초기 반복 평가에 유용합니다.

이 평가자는 세 가지 기준으로 점수를 매겨요:
- 답변 **길이** (너무 짧거나 길면 감점)
- **키워드** 포함 여부 (Transformer 관련 핵심 용어)
- **불필요한 표현** 없음 ("I don't know" 등)

In [ ]:
# ---------------------------------------------------
# 규칙 기반 커스텀 평가자 구현
# ---------------------------------------------------
# ============================================================
# TODO: (Run, Example) -> dict 형식의 규칙 기반 평가자 함수를 완성하세요
# 힌트: run.outputs.get("answer", ""), 길이/키워드/불필요표현 3가지 기준으로 점수 합산
# 예상 결과: {"key": "heuristic_score", "score": 0~1}
# ============================================================

from langsmith.schemas import Run, Example



### 규칙 기반 평가자 테스트

실제 LangSmith 실험 전에 Mock 객체로 동작을 미리 확인해볼게요.

In [ ]:
# ---------------------------------------------------
# Mock 객체로 규칙 기반 평가자 동작 검증
# ---------------------------------------------------
# 테스트용 Mock 객체 생성
class MockRun:
    def __init__(self, outputs):
        self.outputs = outputs

class MockExample:
    def __init__(self, inputs, outputs):
        self.inputs = inputs
        self.outputs = outputs

# ============================================================
# TODO: 위 Mock 객체를 사용해 규칙 기반 평가자를 테스트하세요
# 힌트: MockRun({"answer": "..."})로 좋은/나쁜 답변을 만들고 length_and_keyword_evaluator() 호출
# 예상 결과: 좋은 답변 → 높은 점수, 나쁜 답변 → 낮은 점수
# ============================================================



---

## 2. LLM 기반 커스텀 평가자

LLM을 직접 평가자로 활용해요. 규칙으로 포착하기 어려운 **근거성(Groundedness)** — 답변이 컨텍스트에만 근거하는지 — 을 판단합니다.

> **주의**: LLM 기반 평가자는 매 평가마다 LLM 호출이 발생해서 추가 비용이 들어요. 개발 초기에는 규칙 기반으로 먼저 테스트하고, 정밀 평가 시 LLM 기반을 사용하는 것이 좋아요.

In [ ]:
# ---------------------------------------------------
# LLM 기반 커스텀 평가자 구현 (Groundedness)
# ---------------------------------------------------
# ============================================================
# TODO: LLM 평가 체인을 만들고 0~10 점수를 0~1로 정규화하는 평가자를 완성하세요
# 힌트: evaluation_chain.invoke({...})로 점수 문자열 받고, re.findall(r'\d+', ...)로 숫자 추출
# 예상 결과: {"key": "llm_groundedness", "score": 0~1}
# ============================================================

from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
import openai



---

## 3. 임베딩 거리 기반 평가자

생성된 답변과 참조 답변의 **코사인 유사도(Cosine Similarity)**를 측정해요. LLM 호출보다 저렴하고, 의미적 유사도를 측정할 수 있어서 유용합니다.

참조 답변이 있는 경우에만 사용할 수 있어요.

In [ ]:
# ---------------------------------------------------
# 임베딩 거리 기반 커스텀 평가자 구현
# ---------------------------------------------------
# ============================================================
# TODO: 코사인 유사도를 계산하는 임베딩 평가자를 완성하세요
# 힌트: embedding_model.embed_documents([generated, reference])로 두 벡터를 얻고, np.dot() / (norm * norm)으로 계산
# 예상 결과: {"key": "embedding_similarity", "score": 0~1}
# ============================================================

from langchain_openai import OpenAIEmbeddings
import numpy as np



---

## LangSmith 데이터셋 준비

평가 실행 전에 LangSmith에 데이터셋을 올려요. 이번에는 참조 답변도 포함합니다 (임베딩 기반 평가자가 사용).

In [ ]:
# ---------------------------------------------------
# LangSmith에 평가용 데이터셋 업로드 (질문 + 참조 답변)
# ---------------------------------------------------
from langsmith import Client
from langsmith import utils as ls_utils

client = Client()
dataset_name = "transformer-qa-custom-eval"

# 평가용 데이터 (질문 + 참조 답변)
qa_pairs = [
    {
        "question": "What is the Transformer architecture?",
        "answer": "The Transformer is a neural network architecture that relies entirely on attention mechanisms, without using recurrent or convolutional layers."
    },
    {
        "question": "How does self-attention work?",
        "answer": "Self-attention computes attention weights between all positions in a sequence, allowing the model to jointly attend to information from different representation subspaces."
    },
    {
        "question": "What are the advantages of Transformers over RNNs?",
        "answer": "Transformers can process sequences in parallel, capture long-range dependencies more effectively, and are more efficient to train than RNNs."
    },
    {
        "question": "What is multi-head attention?",
        "answer": "Multi-head attention applies multiple attention mechanisms in parallel, allowing the model to jointly attend to information from different representation subspaces."
    },
]

# 데이터셋 생성 또는 사용
try:
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"✅ 기존 데이터셋 사용: {dataset_name}")
except ls_utils.LangSmithNotFoundError:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Transformer Q&A for custom evaluator testing"
    )
    
    for qa in qa_pairs:
        client.create_example(
            inputs={"question": qa["question"]},
            outputs={"answer": qa["answer"]},
            dataset_id=dataset.id
        )
    print(f"✅ 새 데이터셋 생성: {dataset_name} ({len(qa_pairs)}개 예제)")

---

## 평가 실행

세 가지 커스텀 평가자를 조합해서 종합 평가를 실행합니다.

In [ ]:
# ---------------------------------------------------
# 3가지 커스텀 평가자 조합하여 종합 평가 실행
# ---------------------------------------------------
# ============================================================
# TODO: evaluate()에 세 가지 평가자 리스트를 전달하여 평가를 실행하세요
# 힌트: evaluators=[length_and_keyword_evaluator, llm_groundedness_evaluator, embedding_similarity_evaluator]
# 예상 결과: experiment_results.experiment_url 출력
# ============================================================

from langsmith.evaluation import evaluate



---

## 정리

### 커스텀 평가자 3가지 유형 비교

| 유형 | 장점 | 단점 | 적합한 사용 사례 |
|------|------|------|----------------|
| **규칙 기반** | 빠름, 비용 없음, 투명함 | 단순한 판단만 가능 | 길이, 키워드, 형식 체크 |
| **LLM 기반** | 정교한 판단, 맥락 이해 | 비용 발생, 일관성 낮음 | 근거성, 품질, 논리성 평가 |
| **임베딩 기반** | 의미 유사도, LLM보다 저렴 | 참조 답변 필수 | 정답과의 유사도 측정 |

### 평가자 조합 전략

- **개발 초기 (빠른 반복)**: 규칙 기반만
- **중간 점검 (정밀도 필요)**: 규칙 기반 + 임베딩 기반
- **최종 평가 (모든 측면)**: 세 가지 모두

### 커스텀 평가자 작성 체크리스트

1. `(Run, Example) -> dict` 시그니처 준수
2. `score`는 반드시 0~1 사이 값
3. `run.outputs`와 `example.outputs` 존재 여부 검증
4. 예외 발생 시 0 반환 (평가 중단 방지)
5. `key` 이름을 직관적으로 (`heuristic_score`, `llm_groundedness` 등)

### 다음 노트북 예고

다음에는 LLM 호출 없이 통계적 방법으로 평가하는 **Heuristic 메트릭** — ROUGE, BLEU, METEOR, SemScore — 을 학습해요.